# Run Pipeline

This notebook follows the current local workflow:

1. Extract BTC archives into `data/keyframes/` and `data/map-keyframes/`
2. Import metadata into `data/index/metadata.jsonl`
3. Generate keyframe captions when transcripts are unavailable
4. Train LoRA CLIP by session, resuming between sessions
5. Extract CLIP features for the keyframes
6. Build the local two-level FAISS index
7. Run a local search test and display the results

In [2]:
from pathlib import Path
import os

def _discover_project_root() -> Path:
    env_root = os.getenv('AIC_PROJECT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser()
        if candidate.exists():
            return candidate.resolve()

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'backend').is_dir():
            return candidate.resolve()

    raise RuntimeError('Set AIC_PROJECT_ROOT or open the notebook inside the project folder.')

PROJECT_ROOT = _discover_project_root()
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

Project root: D:\AI\AIC 2026\video-search-agent


In [3]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', '-r', 'D:\\AI\\AIC 2026\\video-search-agent\\requirements.txt'], returncode=0)

## Step 0 - Check the runtime

Use this cell to confirm whether the notebook is attached to an NVIDIA GPU or running on CPU.

In [4]:
import os
import torch

# Fixed profile for NVIDIA GPUs with 8 GB VRAM. Keep these values unchanged.
# The micro-batch controls peak VRAM; gradient accumulation preserves throughput.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

PyTorch: 2.4.1+cpu
CUDA available: False
Device: cpu


## Step 1 - Extract BTC archives

Run this only once per dataset update. It creates the raw `keyframes/` and `map-keyframes/` folders that the rest of the pipeline reads.

In [5]:
# Preview what will be extracted
subprocess.run([sys.executable, 'scripts/extract_btc_data.py', '--dry-run'], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/extract_btc_data.py', '--dry-run'], returncode=0)

In [6]:
# Extract for real
subprocess.run([sys.executable, 'scripts/extract_btc_data.py'], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/extract_btc_data.py'], returncode=0)

## Step 2 - Import metadata

This writes the canonical `data/index/metadata.jsonl` used by training, captioning, feature extraction, and indexing. Keep transcript disabled unless `data/videos/` contains source videos.

In [7]:
# Metadata only. Add '--with-transcript' only when data/videos contains source videos.
subprocess.run([sys.executable, 'scripts/import_btc_data.py', '--force'], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/import_btc_data.py', '--force'], returncode=0)

## Step 3 - Generate keyframe captions

Captions improve LoRA training when transcripts are unavailable. The pipeline captions one representative keyframe per 5-second VFR-safe window, then propagates the caption to that window. Run the smoke test first, then run the full caption step.

In [8]:
CAPTION_TEST_LIMIT = 100
CAPTION_BATCH_SIZE = 2
CAPTION_MODEL = 'Salesforce/blip-image-captioning-base'
CAPTION_MODEL_TYPE = 'blip'
CAPTION_PROMPT = 'aic'
CAPTION_MIN_WORDS = 8
CAPTION_MAX_NEW_TOKENS = 48
CAPTION_NUM_BEAMS = 2
CAPTION_WINDOW_SECONDS = 5.0

print('Caption settings:')
print(f'  test_limit={CAPTION_TEST_LIMIT}')
print(f'  batch_size={CAPTION_BATCH_SIZE}')
print(f'  model={CAPTION_MODEL}')
print(f'  prompt={CAPTION_PROMPT}')
print(f'  min_words={CAPTION_MIN_WORDS}')
print(f'  max_new_tokens={CAPTION_MAX_NEW_TOKENS}')
print(f'  num_beams={CAPTION_NUM_BEAMS}')
print(f'  window_seconds={CAPTION_WINDOW_SECONDS}')

Caption settings:
  test_limit=100
  batch_size=2
  model=Salesforce/blip-image-captioning-base
  prompt=aic
  min_words=8
  max_new_tokens=48
  num_beams=2
  window_seconds=5.0


In [10]:
# Caption smoke test
caption_test_cmd = [
    sys.executable,
    '-m', 'backend.preprocessing.generate_captions',
    '--limit', str(CAPTION_TEST_LIMIT),
    '--batch-size', str(CAPTION_BATCH_SIZE),
    '--model-name', CAPTION_MODEL,
    '--model-type', CAPTION_MODEL_TYPE,
    '--prompt', CAPTION_PROMPT,
    '--min-words', str(CAPTION_MIN_WORDS),
    '--max-new-tokens', str(CAPTION_MAX_NEW_TOKENS),
    '--num-beams', str(CAPTION_NUM_BEAMS),
    '--window-seconds', str(CAPTION_WINDOW_SECONDS),
    '--force',
]
print('Running:', ' '.join(caption_test_cmd))
subprocess.run(caption_test_cmd, cwd=PROJECT_ROOT, check=True)

Running: c:\Users\Administrator\AppData\Local\Programs\Python\Python312\python.exe -m backend.preprocessing.generate_captions --limit 100 --batch-size 2 --model-name Salesforce/blip-image-captioning-base --model-type blip --prompt aic --min-words 8 --max-new-tokens 48 --num-beams 2 --window-seconds 5.0 --force


CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'backend.preprocessing.generate_captions', '--limit', '100', '--batch-size', '2', '--model-name', 'Salesforce/blip-image-captioning-base', '--model-type', 'blip', '--prompt', 'aic', '--min-words', '8', '--max-new-tokens', '48', '--num-beams', '2', '--window-seconds', '5.0', '--force'], returncode=0)

In [ ]:
# Full 5-second representative-window caption run. Skip this if metadata text is sufficient.
caption_full_cmd = [
    sys.executable,
    '-m', 'backend.preprocessing.generate_captions',
    '--batch-size', str(CAPTION_BATCH_SIZE),
    '--model-name', CAPTION_MODEL,
    '--model-type', CAPTION_MODEL_TYPE,
    '--prompt', CAPTION_PROMPT,
    '--min-words', str(CAPTION_MIN_WORDS),
    '--max-new-tokens', str(CAPTION_MAX_NEW_TOKENS),
    '--num-beams', str(CAPTION_NUM_BEAMS),
    '--window-seconds', str(CAPTION_WINDOW_SECONDS),
    '--force',
]
print('Running:', ' '.join(caption_full_cmd))
subprocess.run(caption_full_cmd, cwd=PROJECT_ROOT, check=True)

## Step 4 - Train LoRA by session

Train in short sessions, then resume from the saved checkpoint. That keeps each run manageable and makes it easy to continue later.

Suggested pattern:
- Session 1: train on a smaller subset or fewer epochs
- Session 2+: resume from `data/index/clip-b32-btc-v1/lora_weights.pt`
- Increase `--limit` only when the previous session is stable

In [11]:
import torch

TRAIN_LIMIT = 0
TEST_LIMIT = 100
TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 8
TRAIN_ACCUM_STEPS = 4
TRAIN_NUM_WORKERS = 0
TEST_NUM_WORKERS = 0
FEATURE_BATCH_SIZE = 8
FEATURE_TEST_LIMIT = 100
TRAIN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if (TRAIN_BATCH_SIZE, TRAIN_ACCUM_STEPS, TRAIN_NUM_WORKERS, FEATURE_BATCH_SIZE) != (8, 4, 0, 8):
    raise RuntimeError('The notebook uses a fixed 8 GB VRAM profile; do not change training parameters.')

print('Training settings:')
print(f'  limit={TRAIN_LIMIT} (full train)')
print(f'  test_limit={TEST_LIMIT} (smoke test)')
print(f'  epochs={TRAIN_EPOCHS}')
print(f'  batch_size={TRAIN_BATCH_SIZE}')
print(f'  accum_steps={TRAIN_ACCUM_STEPS}')
print(f'  num_workers={TRAIN_NUM_WORKERS}')
print(f'  device={TRAIN_DEVICE}')
print(f'  feature_batch_size={FEATURE_BATCH_SIZE}')

Training settings:
  limit=0 (full train)
  test_limit=100 (smoke test)
  epochs=1
  batch_size=8
  accum_steps=4
  num_workers=0
  device=cpu
  feature_batch_size=8


In [12]:
# Estimate the full L21-L24 run. This cell only reads CSV metadata.
import csv
import math

target_datasets = ('L21', 'L22', 'L23', 'L24')
map_dir = PROJECT_ROOT / 'data' / 'map-keyframes'
counts = {}
for dataset in target_datasets:
    files = sorted(map_dir.glob(dataset + '_*.csv'))
    counts[dataset] = sum(sum(1 for _ in csv.DictReader(f.open(encoding='utf-8', newline=''))) for f in files)
observed = {name: count for name, count in counts.items() if count > 0}
if not observed:
    print(f'No map-keyframes CSV found under {map_dir}')
else:
    observed_total = sum(observed.values())
    missing = [name for name in target_datasets if name not in observed]
    average_observed = observed_total / len(observed)
    projected_total = observed_total + round(average_observed * len(missing))
    total_keyframes = projected_total if missing else observed_total
    train_samples = max(0, total_keyframes - int(total_keyframes * 0.10))
    micro_steps = math.ceil(train_samples / TRAIN_BATCH_SIZE)
    optimizer_steps = math.ceil(micro_steps / TRAIN_ACCUM_STEPS)
    # Conservative ranges for the fixed 5060 Ti 8 GB profile.
    caption_seconds = (0.40, 1.00)     # BLIP base, batch 2, 48 tokens, beams 2
    train_seconds = (0.35, 0.75)       # CLIP ViT-B/32, micro-batch 8
    feature_seconds = (0.12, 0.30)     # CLIP feature extraction, batch 8
    representative_keyframes = math.ceil(total_keyframes * 0.20)
    caption_hours = tuple(representative_keyframes * value / 3600 for value in caption_seconds)
    train_hours = tuple(micro_steps * value / 3600 for value in train_seconds)
    feature_hours = tuple(math.ceil(total_keyframes / FEATURE_BATCH_SIZE) * value / 3600 for value in feature_seconds)
    print('Estimated run for L21-L24:')
    print(f'  observed keyframes: {observed}')
    print(f'  projected total keyframes: {total_keyframes:,}')
    print(f'  representative caption frames (5s estimate): {representative_keyframes:,}')
    if missing:
        print(f'  projection used for missing datasets: {missing}')
    print(f'  import metadata: approximately 1-10 minutes')
    print(f'  generate captions: {caption_hours[0]:.1f}-{caption_hours[1]:.1f} hours')
    print(f'  LoRA training: {train_hours[0]:.1f}-{train_hours[1]:.1f} hours per epoch')
    print(f'    micro-steps: {micro_steps:,} | optimizer-steps: {optimizer_steps:,}')
    print(f'  extract CLIP features: {feature_hours[0]:.1f}-{feature_hours[1]:.1f} hours')
    print(f'  build two-level FAISS index: approximately 2-15 minutes')
    print('  archive download/extraction: depends on network and disk; measure separately')
    print('Note: captioning remains the dominant step even with the faster BLIP-base profile.')

Estimated run for L21-L24:
  observed keyframes: {'L21': 22248}
  projected total keyframes: 88,992
  representative caption frames (5s estimate): 17,799
  projection used for missing datasets: ['L22', 'L23', 'L24']
  import metadata: approximately 1-10 minutes
  generate captions: 2.0-4.9 hours
  LoRA training: 1.0-2.1 hours per epoch
    micro-steps: 10,012 | optimizer-steps: 2,503
  extract CLIP features: 0.4-0.9 hours
  build two-level FAISS index: approximately 2-15 minutes
  archive download/extraction: depends on network and disk; measure separately
Note: captioning remains the dominant step even with the faster BLIP-base profile.


In [13]:
# Session 0: TEST
import subprocess, sys

train_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TEST_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TEST_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)


Running: c:\Users\Administrator\AppData\Local\Programs\Python\Python312\python.exe scripts/train_lora_clip.py --limit 100 --epochs 1 --batch-size 8 --num-workers 0 --device cpu


CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/train_lora_clip.py', '--limit', '100', '--epochs', '1', '--batch-size', '8', '--num-workers', '0', '--device', 'cpu'], returncode=0)

In [ ]:
# Session 1: fresh training run
# Increase --epochs if you want a longer first session.
import subprocess, sys

train_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TRAIN_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--accum-steps', str(TRAIN_ACCUM_STEPS),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Session 2: resume from the saved checkpoint
# Run this after Session 1 if you want to continue training in another pass.
import subprocess, sys

resume_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TRAIN_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--accum-steps', str(TRAIN_ACCUM_STEPS),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
    '--resume',
]
print('Running:', ' '.join(resume_cmd))
subprocess.run(resume_cmd, cwd=PROJECT_ROOT, check=True)


## Step 5 - Extract CLIP features

This reads your current keyframes and writes `.npy` vectors to `data/clip-features/`.

In [14]:
# Smoke test 100 keyframes before the full feature extraction run.
feature_test_cmd = [
    sys.executable,
    'scripts/extract_clip_features.py',
    '--num-workers', str(TEST_NUM_WORKERS),
    '--batch-size', str(FEATURE_BATCH_SIZE),
    '--limit', str(FEATURE_TEST_LIMIT),
]
print('Running:', ' '.join(feature_test_cmd))
subprocess.run(feature_test_cmd, cwd=PROJECT_ROOT, check=True)

Running: c:\Users\Administrator\AppData\Local\Programs\Python\Python312\python.exe scripts/extract_clip_features.py --num-workers 0 --batch-size 8 --limit 100


CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/extract_clip_features.py', '--num-workers', '0', '--batch-size', '8', '--limit', '100'], returncode=0)

### Step 5.1 - Full feature extraction
Run this after the 100-keyframe smoke test passes and the final LoRA checkpoint is ready.

In [ ]:
feature_cmd = [
    sys.executable,
    'scripts/extract_clip_features.py',
    '--num-workers', str(TEST_NUM_WORKERS),
    '--batch-size', str(FEATURE_BATCH_SIZE),
]
print('Running:', ' '.join(feature_cmd))
subprocess.run(feature_cmd, cwd=PROJECT_ROOT, check=True)

## Step 6 - Build the local FAISS index

This creates the complete bundle in `data/index/clip-b32-btc-v1/`: `artifact_manifest.json`, `video.index`, `index_metadata.json`, and (when enabled) `lora_weights.pt`.

In [ ]:
subprocess.run([sys.executable, '-m', 'backend.embedding.build_index'], cwd=PROJECT_ROOT, check=True)

## Step 7 - Local search test

This is the notebook-style test flow similar to the old version: encode a query, search the local FAISS index, and print the top hits.

In [ ]:
import json
from pathlib import Path

import faiss
import numpy as np

from backend.config import FAISS_INDEX_PATH, FAISS_METADATA_PATH
from backend.embedding.clip_encoder import encode_text

In [ ]:
index = faiss.read_index(str(FAISS_INDEX_PATH))
with open(FAISS_METADATA_PATH, encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Loaded {index.ntotal} vectors and {len(metadata)} metadata rows.')

In [ ]:
query = 'a photo of a tree'
vec = encode_text(query).reshape(1, -1).astype(np.float32)
faiss.normalize_L2(vec)

top_k = 10
scores, indices = index.search(vec, top_k)

print(f"Query: {query}")
for i, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    if idx < 0 or idx >= len(metadata):
        continue
    item = metadata[idx]
    print(f"#{i:02d} | score={score:.4f} | {item.get('video_id', '')} | frame={item.get('frame_id', '')} | pts={item.get('pts_time', 0.0):.2f}s")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

n = min(top_k, 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(f"Search results: {query}")

for i, ax in enumerate(axes.flat):
    if i >= n:
        ax.axis('off')
        continue
    idx = indices[0][i]
    if idx < 0 or idx >= len(metadata):
        ax.axis('off')
        continue

    item = metadata[idx]
    img_path = Path(item.get('path', ''))
    if img_path.exists():
        ax.imshow(Image.open(img_path).convert('RGB'))
    else:
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=ax.transAxes)

    ax.set_title(f"#{i+1} | {item.get('video_id', '')} | {item.get('frame_id', '')}")
    ax.axis('off')

plt.tight_layout()
plt.show()